# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04** </center>
---
**Profesor**: Pablo Camarillo Ramirez

Lab 04 - Daniel Sebastian Macias

In [1]:
from sebastianM.spark_utils import SparkUtils
from pyspark.sql.functions import get_json_object, col

su = SparkUtils("Lab 04", "spark://spark-cluster-spark-master-1:7077")
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 21:01:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
agencies_schema = SparkUtils.generate_schema([("agency_id", "int"),("agency_info", "string")])
brands_schema = SparkUtils.generate_schema([("brand_id", "int"),("brand_info", "string")])
car_schema = SparkUtils.generate_schema([("car_id", "int"),("car_info", "string")])
customer_schema = SparkUtils.generate_schema([("customer_id", "int"),("customer_info", "string")])
rental_schema = SparkUtils.generate_schema([("rental_id", "int"),("rental_info", "string")])

In [3]:
agencies_df = su._spark.read.schema(agencies_schema).option("header", "true").csv("/opt/spark/work-dir/data/car_service/agencies")
brands_df = su._spark.read.schema(brands_schema).option("header", "true").csv("/opt/spark/work-dir/data/car_service/brands")
car_df = su._spark.read.schema(car_schema).option("header", "true").csv("/opt/spark/work-dir/data/car_service/cars")
customer_df = su._spark.read.schema(customer_schema).option("header", "true").csv("/opt/spark/work-dir/data/car_service/customers")
rentals_df = su._spark.read.schema(rental_schema).option("header", "true").csv("/opt/spark/work-dir/data/car_service/rentals")

In [4]:
car_df = car_df.withColumn("car_name", get_json_object(car_df.car_info, "$.car_name"))
agencies_df = agencies_df.withColumn("agency_name", get_json_object(agencies_df.agency_info, "$.agency_name"))
customer_df = customer_df.withColumn("customer_name", get_json_object(customer_df.customer_info, "$.customer_name"))
rentals_df = rentals_df \
    .withColumn("car_id",      get_json_object(rentals_df.rental_info, "$.car_id").cast("int")) \
    .withColumn("agency_id",   get_json_object(rentals_df.rental_info, "$.agency_id").cast("int")) \
    .withColumn("customer_id", get_json_object(rentals_df.rental_info, "$.customer_id").cast("int"))

In [5]:
df = rentals_df.join(car_df, on="car_id") \
                        .join(agencies_df, on="agency_id") \
                        .join(customer_df, on="customer_id") \
                        .select("rental_id", "car_name", "agency_name", "customer_name")
df.show(5, truncate=False)

+---------+-----------------------+-------------+---------------+
|rental_id|car_name               |agency_name  |customer_name  |
+---------+-----------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9|NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8   |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5  |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4     |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1     |SF Cars      |Kristin Potts  |
+---------+-----------------------+-------------+---------------+
only showing top 5 rows


In [9]:
agencies_df.count()

5

In [12]:
agencies_df.show()
brands_df.show()
car_df.show()
customer_df.show()
rentals_df.show()

+---------+--------------------+-------------+
|agency_id|         agency_info|  agency_name|
+---------+--------------------+-------------+
|        1|{'agency_name': '...|  NYC Rentals|
|        2|{'agency_name': '...|LA Car Rental|
|        3|{'agency_name': '...| Zapopan Auto|
|        4|{'agency_name': '...|      SF Cars|
|        5|{'agency_name': '...|  Mexico Cars|
+---------+--------------------+-------------+

+--------+--------------------+
|brand_id|          brand_info|
+--------+--------------------+
|       1|{'brand_name': 'M...|
|       2|{'brand_name': 'B...|
|       3|{'brand_name': 'A...|
|       4|{'brand_name': 'F...|
|       5|{'brand_name': 'B...|
|       6|{'brand_name': 'H...|
|       7|{'brand_name': 'T...|
+--------+--------------------+

+------+--------------------+--------------------+
|car_id|            car_info|            car_name|
+------+--------------------+--------------------+
|     1|{'car_name': 'Cha...|Chang-Fisher Model 7|
|     2|{'car_name'

In [14]:
agencies_df.write.mode("overwrite") \
    .partitionBy("agency_id") \
    .parquet("data/car_service_parquet/agencies")

brands_df.write.mode("overwrite") \
    .partitionBy("brand_id") \
    .parquet("data/car_service_parquet/brands")

car_df.write.mode("overwrite") \
    .partitionBy("car_id") \
    .parquet("data/car_service_parquet/cars")

customer_df.write.mode("overwrite") \
    .partitionBy("customer_id") \
    .parquet("data/car_service_parquet/customers")

rentals_df.write.mode("overwrite") \
    .partitionBy("agency_id") \
    .parquet("data/car_service_parquet/rentals")

In [18]:
import os

print("Brands:", os.listdir("data/car_service_parquet/brands"))
print("Agencies:", os.listdir("data/car_service_parquet/agencies"))
print("Cars:", os.listdir("data/car_service_parquet/cars"))
print("Customers:", os.listdir("data/car_service_parquet/customers"))
print("Rentals:", os.listdir("data/car_service_parquet/rentals"))

Brands: ['._SUCCESS.crc', 'brand_id=1', 'brand_id=2', 'brand_id=3', 'brand_id=4', 'brand_id=5', 'brand_id=6', 'brand_id=7', '_SUCCESS']
Agencies: ['._SUCCESS.crc', 'agency_id=1', 'agency_id=2', 'agency_id=3', 'agency_id=4', 'agency_id=5', '_SUCCESS']
Cars: ['._SUCCESS.crc', 'car_id=1', 'car_id=10', 'car_id=11', 'car_id=12', 'car_id=13', 'car_id=14', 'car_id=15', 'car_id=16', 'car_id=17', 'car_id=18', 'car_id=19', 'car_id=2', 'car_id=20', 'car_id=21', 'car_id=22', 'car_id=23', 'car_id=24', 'car_id=25', 'car_id=26', 'car_id=27', 'car_id=28', 'car_id=29', 'car_id=3', 'car_id=4', 'car_id=5', 'car_id=6', 'car_id=7', 'car_id=8', 'car_id=9', '_SUCCESS']
Customers: ['._SUCCESS.crc', 'customer_id=1', 'customer_id=10', 'customer_id=100', 'customer_id=101', 'customer_id=102', 'customer_id=103', 'customer_id=104', 'customer_id=105', 'customer_id=106', 'customer_id=107', 'customer_id=108', 'customer_id=109', 'customer_id=11', 'customer_id=110', 'customer_id=111', 'customer_id=112', 'customer_id=113